# Heretic TPU Environment Check

Run this BEFORE the real heretic notebook. Verifies:
1. TPU detection + basic ops
2. Heretic install + dependency compatibility
3. XLA-compatible forward pass (logits, residuals)
4. XLA-compatible greedy generation
5. Full model load + single trial dry run

All cells should pass before running the ablation notebook.

In [ ]:
# CELL 1: Install Heretic + Dependencies
# ~5 min

%pip uninstall -q -y numba llvmlite scipy scikit-learn numpy 2>/dev/null || true
!rm -rf /usr/local/lib/python3.12/dist-packages/numpy* /usr/local/lib/python3.12/dist-packages/scipy*

%pip install -q --no-cache-dir numpy==2.2.2
%pip install -q "heretic-llm[tpu] @ git+https://github.com/instax-dutta/heretic.git"
%pip install -q scipy==1.14.1 --no-deps
%pip install -q scikit-learn --no-deps joblib threadpoolctl

import numpy
assert hasattr(numpy._core.umath, '_center'), 'numpy C extensions stale - restart runtime'
print(f'numpy {numpy.__version__} OK')

In [ ]:
# CELL 2: TPU Detection + Basic Tensor Ops

import torch, torch_xla, torch_xla.core.xla_model as xm
import os, sys

print(f"PyTorch:   {torch.__version__}")
print(f"torch_xla: {torch_xla.__version__}")
print(f"Python:    {sys.version.split()[0]}")
print(f"PJRT_DEVICE: {os.environ.get('PJRT_DEVICE', 'NOT SET')}")

device = torch_xla.device()
print(f"Device:    {device}")
print(f"HW:        {xm.xla_device_hw(device)}")

# Basic matmul
x = torch.randn(128, 128, device=device, dtype=torch.bfloat16)
y = torch.randn(128, 128, device=device, dtype=torch.bfloat16)
z = x @ y
torch_xla.sync()
print(f"Matmul:    {z.shape} {z.dtype} on {z.device} OK")

# Test bf16 range
big = torch.full((1,), 65504.0, device=device, dtype=torch.bfloat16)
torch_xla.sync()
print(f"bf16 max:  {big.item()} OK")

print("\nTPU basics PASS")

In [ ]:
# CELL 3: Heretic Import + TPU Detection

from heretic.system import detect_tpu, setup_tpu_environment, mark_step
from heretic.config import Settings
from heretic.model import Model

is_tpu = detect_tpu()
print(f"detect_tpu(): {is_tpu}")
assert is_tpu, "TPU not detected - check runtime type"

setup_tpu_environment()
print("setup_tpu_environment() OK")

print("Heretic imports PASS")

In [ ]:
# CELL 4: Config Validation on TPU

import sys as _sys

_sys.argv = [
    'heretic',
    '--model=Qwen/Qwen2.5-Coder-0.5B-Instruct',
    '--dtypes=bfloat16',
    '--quantization=none',
    '--device-map=auto',
    '--n-trials=1',
    '--n-startup-trials=1',
    '--max-response-length=10',
    '--row-normalization=full',
    '--full-normalization-lora-rank=3',
    '--orthogonalize-direction',
    '--winsorization-quantile=1.0',
    '--offload-outputs-to-cpu',
]

settings = Settings()
print(f"Model:       {settings.model}")
print(f"Dtypes:      {settings.dtypes}")
print(f"Quantization:{settings.quantization}")
print(f"Device map:  {settings.device_map}")
print(f"Batch size:  {settings.batch_size}")
print(f"TPU cores:   {settings.tpu_cores}")
print(f"BF16 forced: {'bfloat16' in [str(d) for d in settings.dtypes]}")
assert settings.quantization == 'none', 'Quantization must be none on TPU'

print("\nConfig validation PASS")

In [ ]:
# CELL 5: Model Load + Forward Pass Test
# ~2 min (downloads 135M instruct model)

from heretic.utils import Prompt

# SmolLM2-135M lacks a chat template; use the Instruct variant.
import sys as _sys
_sys.argv = [
    'heretic',
    '--model=Qwen/Qwen2.5-Coder-0.5B-Instruct',
    '--dtypes=bfloat16',
    '--quantization=none',
    '--device-map=auto',
    '--n-trials=1',
    '--n-startup-trials=1',
    '--max-response-length=10',
    '--row-normalization=full',
    '--full-normalization-lora-rank=3',
    '--orthogonalize-direction',
    '--winsorization-quantile=1.0',
    '--offload-outputs-to-cpu',
    '--batch-size=2',
]
settings = Settings()

m = Model(settings)
print(f"Model loaded: {type(m.model).__name__}")
print(f"Model device: {m.model.device}")
print(f"Model dtype:  {next(m.model.parameters()).dtype}")
print(f"Chat template: {'set' if m.tokenizer.chat_template else 'MISSING'}")

# Test forward pass (XLA-compatible path)
test_prompts = [
    Prompt(system="You are a helpful assistant.", user="What is 2+2?"),
    Prompt(system="You are a helpful assistant.", user="Hello!"),
]

print("\nTesting forward()...")
inputs, outputs = m.forward(test_prompts, use_cache=False)
print(f"  Logits shape: {outputs.logits.shape}")
print(f"  Logits device: {outputs.logits.device}")
assert outputs.logits.device.type in ('xla', 'tpu'), f"Logits on wrong device: {outputs.logits.device}"
print("  forward() PASS")

print("\nModel load + forward PASS")

In [ ]:
# CELL 6: Test get_logits() on TPU

print("Testing get_logits()...")
logits = m.get_logits(test_prompts)
print(f"  Shape: {logits.shape}")
print(f"  Device: {logits.device}")
print(f"  dtype: {logits.dtype}")
print(f"  Min: {logits.min().item():.4f}  Max: {logits.max().item():.4f}")
assert logits.shape[0] == 2, f"Expected batch=2, got {logits.shape[0]}"
assert logits.shape[1] > 0, "Empty vocab"
print("  get_logits() PASS")

In [ ]:
# CELL 7: Test get_residuals() on TPU

print("Testing get_residuals()...")
residuals = m.get_residuals(test_prompts)
print(f"  Shape: {residuals.shape}")  # (batch, layers, hidden)
print(f"  Device: {residuals.device}")
print(f"  dtype: {residuals.dtype}")
assert residuals.shape[0] == 2, f"Expected batch=2, got {residuals.shape[0]}"
assert residuals.shape[1] > 0, "No layers"
print(f"  Layers: {residuals.shape[1]}")
print(f"  Hidden: {residuals.shape[2]}")
print("  get_residuals() PASS")

In [ ]:
# CELL 8: Test get_responses() on TPU (XLA greedy loop)

print("Testing get_responses() (XLA greedy generation)...")
responses = m.get_responses(test_prompts)
print(f"  Batch size: {len(responses)}")
for i, r in enumerate(responses):
    print(f"  Response {i}: '{r[:80]}{'...' if len(r)>80 else ''}'")
assert len(responses) == 2, f"Expected 2 responses, got {len(responses)}"
assert all(isinstance(r, str) for r in responses), "Non-string responses"
print("  get_responses() PASS")

In [ ]:
# CELL 9: XLA Compilation + Re-tracing Test
# Run forward pass 3x to verify XLA caches the compiled graph

import time

print("Testing XLA compilation caching (3 iterations)...")
times = []
for i in range(3):
    t0 = time.time()
    logits = m.get_logits(test_prompts)
    dt = time.time() - t0
    times.append(dt)
    print(f"  Iter {i+1}: {dt:.2f}s")

if times[1] < times[0] * 0.8:
    print(f"  XLA caching working ({times[0]:.2f}s -> {times[1]:.2f}s)")
else:
    print(f"  Warning: no speedup from XLA caching")
print("  XLA compilation test DONE")

In [ ]:
# CELL 10: KL Divergence Scorer Test

from heretic.scorers.kl_divergence import KLDivergence
from heretic.plugin import Context

print("Testing KLDivergence scorer...")
kl_scorer = KLDivergence(
    heretic_settings=settings,
    settings=KLDivergence.get_settings_model()(prompts={
        'dataset': 'mlabonne/harmless_alpaca',
        'split': 'test[:2]',
        'column': 'text',
    }),
)

ctx = Context(settings=settings, model=m)
kl_scorer.init(ctx)
baseline_score = kl_scorer.get_baseline_score(ctx)
print(f"  Baseline KL: {baseline_score.rich_display}")
score = kl_scorer.get_score(ctx)
print(f"  Current KL: {score.rich_display}")
print("  KLDivergence scorer PASS")

In [ ]:
# CELL 11: Memory Report

import gc
gc.collect()

print("=" * 60)
print("MEMORY STATUS")
print("=" * 60)

# TPU memory
try:
    import torch_xla.runtime as xr
    mem_info = xr.memory_info()
    print(f"TPU HBM total:     {mem_info['bytes_total'] / 1e9:.2f} GB")
    print(f"TPU HBM in use:    {mem_info['bytes_in_use'] / 1e9:.2f} GB")
    print(f"TPU HBM free:      {(mem_info['bytes_total'] - mem_info['bytes_in_use']) / 1e9:.2f} GB")
except Exception as e:
    print(f"TPU memory info: {e}")
    print("(Not critical - TPU memory is managed by XLA runtime)")

# System RAM
import os
with open('/proc/meminfo') as f:
    for line in f:
        if any(k in line for k in ['MemTotal', 'MemAvailable', 'MemFree']):
            parts = line.split()
            print(f"RAM {parts[0].rstrip(':')}: {int(parts[1]) / 1e6:.1f} GB")

print("\n" + "=" * 60)
print("ALL CHECKS PASSED - Ready for heretic ablation")
print("=" * 60)